In [1]:
## Section 2: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 21 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

SyntaxError: invalid syntax (2397012699.py, line 3)

In [ ]:
# Pretraining notebook

In [1]:

import torch

torch.manual_seed(123)

from weird_ai.model import WeirdAIModel
from weird_ai.tokenizer import SimpleCharacterTokenizer
from weird_ai.generation import (
    text_to_token_ids,
    token_ids_to_text,
    generate_text_simple
)
from weird_ai.config import SAMPLE_LYRICS_FILE

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

with open(SAMPLE_LYRICS_FILE, "r", encoding="utf-8") as f:
    corpus = f.read()

tokenizer = SimpleCharacterTokenizer(corpus)

vocab_size = len(tokenizer.chars)

model = WeirdAIModel(vocab_size)
model.to(device)
model.eval()

prompts = [
    "love is",
    "baby",
    "tonight"
]

for prompt in prompts:
    input_ids = text_to_token_ids(
        prompt,
        tokenizer
    ).to(device)

    with torch.no_grad():
        output_ids = generate_text_simple(
            model,
            input_ids,
            max_new_tokens=21,
            context_size=256
        )

    output_text = token_ids_to_text(
        output_ids,
        tokenizer
    )

    print(f"\nPrompt: {prompt}")
    print(output_text)

cuda

Prompt: love is
love is`é(”넌머곁넌척表X<多넌â~꽤ý都척넌

Prompt: baby
babyé 下넌파넌척넌머곁넌결¡블된배고â~꽤ý

Prompt: tonight
tonight6놀要まй版倒都던젠스が배원ð견がý都¡嘴


In [ ]:
## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [2]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


In [ ]:
## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [3]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

In [4]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.7000)


In [5]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.3567)


In [6]:
loss = -log_probability

print(loss)

tensor(0.3567)


In [7]:
logits = torch.log(probs).unsqueeze(0)
target = torch.tensor([target_index])

torch_loss = torch.nn.functional.cross_entropy(
    logits,
    target
)

print("PyTorch loss:", torch_loss)

PyTorch loss: tensor(0.3567)


In [ ]:
## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [8]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


In [ ]:
## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

In [ ]:
# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [9]:
from weird_ai.dataset import LyricsDataset

train_ratio = 0.9

split_index = int(len(corpus) * train_ratio)

train_text = corpus[:split_index]
val_text = corpus[split_index:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

Training characters: 12156965
Validation characters: 1350774


In [ ]:
block_size = 256

train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

train_dataset = LyricsDataset(
    train_tokens,
    block_size
)

val_dataset = LyricsDataset(
    val_tokens,
    block_size
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

from torch.utils.data import DataLoader

batch_size = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(train_loader)
print(val_loader)

Training samples: 12156709
Validation samples: 1350518


In [11]:
from weird_ai.losses import calc_loss_loader

initial_train_loss = calc_loss_loader(
    train_loader,
    model,
    device
)

initial_val_loss = calc_loss_loader(
    val_loader,
    model,
    device
)

print(f"Initial training loss: {initial_train_loss}")
print(f"Initial validation loss: {initial_val_loss}")

[W921 16:47:21.225200473 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 100663296 bytes (free: 103743488, total: 12541493248).
[W921 16:47:21.238340106 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 100663296 bytes (free: 128909312, total: 12541493248).


OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 11.68 GiB of which 122.94 MiB is free. Process 3497 has 82.59 MiB memory in use. Process 18954 has 203.65 MiB memory in use. Including non-PyTorch memory, this process has 10.00 GiB memory in use. Of the allocated memory 9.81 GiB is allocated by PyTorch, and 70.86 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [ ]:
num_epochs = 1

# TODO:
# Train the model

# TODO:
# Record:
# Training loss before
# Training loss after

In [ ]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

In [ ]:
## Evaluation

Did the loss decrease?

Why is decreasing loss important?

In [ ]:
## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?